In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 128,
    "architecture": "alexnet",
    "learning_rate": 0.001,
    "epochs": 10,
    "pretrained":True
}

In [4]:
transform = transforms.Compose([
    transforms.Resize((model_config["input_size"], model_config["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dir = "../flowers/train"
val_dir = "../flowers/val"

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [5]:
alexnet = models.alexnet(pretrained=True)

for param in alexnet.parameters():
    param.requires_grad = False

alexnet.classifier[-1] = nn.Linear(4096, 5)

for param in alexnet.classifier[-1].parameters():
    param.requires_grad = True

gpu = torch.device("cuda")
alexnet = alexnet.to(gpu)


/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet.parameters(), lr=model_config["learning_rate"])
epochs = model_config["epochs"]

In [7]:
def train_model(model, train_dataloader, optimizer, criterion, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}")

In [8]:
train_model(alexnet, train_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.7676485712487859
Epoch: 2, Training Loss: 0.4621453360168345
Epoch: 3, Training Loss: 0.35901000390786575
Epoch: 4, Training Loss: 0.319202924815129
Epoch: 5, Training Loss: 0.321112791388444
Epoch: 6, Training Loss: 0.2725890990146306
Epoch: 7, Training Loss: 0.25426577635673414
Epoch: 8, Training Loss: 0.24128659806804234
Epoch: 9, Training Loss: 0.2459440791267026
Epoch: 10, Training Loss: 0.21172764662382346


In [10]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [12]:
accuracy = validate_model(alexnet, val_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 89.61424332344214
